## 📂 Querying Files (Lectura de Datos desde Volúmenes)
### 🐍 PySpark + Volúmenes

Otra de las las formas consultar archivos almacenados en un **Volumen** es utilizando **PySpark**. Para ello, disponemos de dos formas:

---

#### 📌 Forma 1: Lectura directa desde el formato del archivo

Esta forma permite consultar directamente los archivos almacenados dentro del volumen indicando el formato del archivo.

```pythom
spark.read.formato_archivo("path_volumen_formato_archivo")
```

#### 📌 Forma 2: Utilizando `load()` especificando el formato

Esta forma permite consultar los archivos almacenados dentro del volumen indicando el formato del archivo previamente.

```python
spark.read.format("formato_archivo").load("path_volumen_formato_archivo")
```

En ambas opciones de lectura, permite especificar ciertas opciones según el tipo de archivo que estemos procesando, proporcionando una mayor granularidad.

---

### 🔍 ¿Cuál es la diferencia?

Ambos enfoques permiten leer archivos almacenados en un Volumen mediante PySpark.

Sin embargo, la forma mas utilizada dependerá de como nos acostumbremos. Mi recomendación es utilizar la forma 2.

---

### 📂 Uso de Wildcards

Ambas formas permiten utilizar **Wildcards (`*`)** para leer múltiples archivos de un mismo Volumen. Algunos escenarios comunes son:

* 📄 Leer todos los archivos del Volumen: 
`path_volumen_formato_archivo`

* 📄 Leer únicamente archivos de una determinada extensión: `path_volumen_formato_archivo/*.extension_archivo`

* 📄 Leer archivos que comiencen con un prefijo específico: `path_volumen_formato_archivo/prefijo-*.extension_archivo`

Gracias a los Wildcards es posible filtrar fácilmente qué archivos serán procesados, permitiendo adaptar la lectura según la organización existente dentro del Volumen.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("12QueryingFilesParte2").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

In [0]:
### ============== CONFIGURACIÓN PREVIA PARA LECTURA CON PYSPARK ============================= ###

### SEGUIREMOS UTILIZANDO EL MISMO VOLUMEN PARA LOS EJEMPLOS DE LECTURA

catalog = "catalog_databricks_2026_de"
schema = "schema_databricks_2026_de"
volumen = "source_data"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volumen}")
print("Volumen creado correctamente")


### SEGUIREMOS UTILIZANDO LAS CARPETAS DENTRO DEL VOLUMEN (CSV y JSON)

path_datasets_csv = f"/Volumes/{catalog}/{schema}/{volumen}/csv" 
path_datasets_json = f"/Volumes/{catalog}/{schema}/{volumen}/json"
path_datasets_binary = f"/Volumes/{catalog}/{schema}/{volumen}/binary"
path_datasets_parquet = f"/Volumes/{catalog}/{schema}/{volumen}/parquet"

dbutils.fs.mkdirs(path_datasets_csv) ## CSV
dbutils.fs.mkdirs(path_datasets_json) ## JSON
dbutils.fs.mkdirs(path_datasets_binary) ## BINARY
dbutils.fs.mkdirs(path_datasets_parquet) ## PARQUET
print("Carpetas CSV, JSON, BINARY y PARQUET creadas correctamente")


### SEGUIREMOS UTILIZNDO LOS DATOS CARGADOS DE STORAGE CLOUD
dbutils.fs.cp(source="s3://bucket-brayan-datasets/csv_datasets/",dest=path_datasets_csv,recurse=True)
dbutils.fs.cp(source="s3://bucket-brayan-datasets/json_datasets/",dest=path_datasets_json,recurse=True)
dbutils.fs.cp(source="s3://bucket-brayan-datasets/binary_datasets/",dest=path_datasets_binary,recurse=True)
dbutils.fs.cp(source="s3://bucket-brayan-datasets/parquet_datasets/",dest=path_datasets_parquet,recurse=True)
print("Archivos copiados exitosamente")


#### ========= CSV =============

In [0]:
### ============== LECTURAS CON PYSPARK ============================= ###

#### A). FORMA 1: LECTURA DIRECTA DESDE EL FORMATO DEL ARCHIVO (sin wildcard)

df = spark.read.option("header","true") \
     .option("inferSchema","true") \
     .csv("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_1.csv")

display(df)

#### B). FORMA 1: LECTURA DIRECTA DESDE EL FORMATO DEL ARCHIVO (con wildcard)

df = spark.read.option("header","true") \
     .option("inferSchema","true") \
     .csv("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_*.csv")

display(df)

#### C). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (sin wildcard)

df = spark.read.format("csv") \
     .option("header","true") \
     .option("inferSchema","true") \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_1.csv")

display(df)

#### D). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (con wildcard)

df = spark.read.format("csv") \
     .option("header","true") \
     .option("inferSchema","true") \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_*.csv")

display(df)


#### ========= JSON =============

In [0]:
### ============== LECTURAS CON PYSPARK ============================= ###

"""
    💡 Los JSON pueden ser de una sola linea o tener múltiples líneas.
        En estos ejemplos, manejaremos ambos tipos.
    💡 Los ejemplos de lectura JSON serán básicos. En próximos capítulos
        profudizaré en las diferentes formas de lectura JSON.
"""

#### A). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (sin wildcard)

df = spark.read \
     .json("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_oneline_1.json")

display(df)

#### B). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (con wildcard)

df = spark.read \
    .json("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_oneline_*.json")

display(df)

#### C). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (sin wildcard)

from pyspark.sql.types import StructType, TimestampType, StringType,IntegerType,StructField

## Inferimos el esquema del JSON (Importante para la lectura JSON multi-linea)
json_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("user_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("action", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("ip_address", StringType(), True),
    StructField("region", StringType(), True),
    StructField("error_code", StringType(), True),  # Se asume StringType por defecto al estar nulo
    StructField("log_level", StringType(), True)
])

## .schema(): Permite agregar el esquema JSON para la lectura fácil.
df = spark.read.format("json") \
     .schema(json_schema) \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_multiline_1.json")

display(df)

#### D). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (con wildcard)

## Inferimos el esquema del JSON (Importante para la lectura JSON multi-linea)
json_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("user_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("action", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("response_time_ms", IntegerType(), True),
    StructField("ip_address", StringType(), True),
    StructField("region", StringType(), True),
    StructField("error_code", StringType(), True),  # Se asume StringType por defecto al estar nulo
    StructField("log_level", StringType(), True)
])

## .schema(): Permite agregar el esquema JSON para la lectura fácil.
df = spark.read.format("json") \
     .schema(json_schema) \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_multiline_*.json")
display(df)

#### ========= BINARIO =============

In [0]:
### ============== LECTURAS CON PYSPARK ============================= ###

#### 💡 La FORMA 1 .binaryFile() no está implementada en Databricks como tal.
#### 💡 Mejor directamente la FORMA 2 .load() 

#### A). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (sin wildcard)

df = spark.read.format("binaryFile") \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/binary/streamingprojecta.png")

display(df)

#### B). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (con wildcard)

df = spark.read.format("binaryFile") \
     .load('/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/binary/Perspective-*.png')

display(df)

#### Existe un limite en el display() al mostrar datos. Por ello sale: Results too large, pero, la lectura fue exitosa.

#### ========= PARQUET (FORMATO COLUMNAR OPTIMIZADO) =============

In [0]:
### ============== LECTURAS CON PYSPARK ============================= ###

#### A). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (sin wildcard)

df = spark.read \
     .parquet("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/infraestructura_costos.parquet")

display(df)


#### B). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (con wildcard)

df = spark.read \
     .parquet("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/logs_actividad_*.parquet")

display(df)


#### C). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (sin wildcard)

df = spark.read \
     .format("parquet") \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/infraestructura_costos.parquet")

display(df)

#### D). FORMA 2: UTILIZANDO .LOAD() ESPECIFICANDO EL FORMATO (con wildcard)

df = spark.read \
     .format("parquet") \
     .load("/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/logs_actividad_*.parquet")

display(df)